# Lab Work - 5.5

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.impute import KNNImputer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from imblearn.over_sampling import SMOTE, ADASYN
from imblearn.under_sampling import RandomUnderSampler  
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')

# Original Dataset
data = {
    'Sample': ['A','B','C','D','E','F','G','H','P','Q','R','S'],
    'x1': [2,3,4,5,6,7,8,9,1,2,3,4],
    'x2': [3,4,3,np.nan,4,3,5,4,1,2,2,1],
    'y': [0,0,0,0,0,0,0,0,1,1,1,1]  # 0=Majority, 1=Minority
}
df = pd.DataFrame(data)
print("Original Dataset:")
display(df)

## Q1: Under-Sampling & Over-Sampling

In [ ]:
# 01 Class Imbalance
n_maj = (df['y'] == 0).sum()
n_min = (df['y'] == 1).sum()
IR = n_maj / n_min
print(f"Majority: {n_maj}, Minority: {n_min}, Imbalance Ratio (IR): {IR:.1f}:1")

In [ ]:
# 02 Random Under-Sampling to 4:4
np.random.seed(42)
maj_to_remove = df[df['y']==0].sample(4, random_state=42).index
under_df = df.drop(maj_to_remove).reset_index(drop=True)
print("Under-sampled Dataset:")
display(under_df)
print("New IR:", (under_df['y']==0).sum() / (under_df['y']==1).sum())

**Risks of Under-sampling**: Loss of information, higher variance.

In [ ]:
# 04 Random Over-Sampling
minority = df[df['y']==1]
over_df = pd.concat([df, minority.sample(4, replace=True, random_state=42)]).reset_index(drop=True)
print("Over-sampled shape:", over_df.shape)
print(over_df['y'].value_counts())

**Risk of Over-sampling**: Overfitting to duplicated points.

## Q2: SMOTE

In [ ]:
# SMOTE Example
X = df[['x1', 'x2']]
y = df['y']
smote = SMOTE(k_neighbors=2, random_state=42)
X_res, y_res = smote.fit_resample(X, y)
print("After SMOTE:", X_res.shape, y_res.value_counts())

In [ ]:
# Manual synthetic example as per Q
P = np.array([1,1])
Q = np.array([2,2])
lam = 0.6
syn = P + lam * (Q - P)
print("Synthetic from P using Q:", syn)

## Q3: ADASYN

In [ ]:
# ADASYN
adasyn = ADASYN(random_state=42)
X_ada, y_ada = adasyn.fit_resample(X, y)
print("After ADASYN:", X_ada.shape, y_ada.value_counts())

## Q4: KNN Imputer + Full Pipeline

In [ ]:
# KNN Imputer for missing x2 in D
imputer = KNNImputer(n_neighbors=3)
X_imputed = pd.DataFrame(imputer.fit_transform(df[['x1','x2']]), columns=['x1','x2'])
print("Imputed Dataset:")
display(pd.concat([X_imputed, df[['Sample','y']]], axis=1))

In [ ]:
# Full Pipeline (KNNImputer + SMOTE + LogisticRegression)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42)

pipeline = ImbPipeline([
    ('imputer', KNNImputer(n_neighbors=3)),
    ('smote', SMOTE(random_state=42)),
    ('classifier', LogisticRegression())
])

pipeline.fit(X_train, y_train)
y_pred_proba = pipeline.predict_proba(X_test)[:, 1]
print("AUC-ROC:", roc_auc_score(y_test, y_pred_proba))

**Golden Rule**: Resampling (SMOTE/ADASYN) **only** on training set to prevent data leakage.